# Pancake Stayman

**Research question:** When Opener bids 1NT and Responder holds a flat hand with a matching 4-card major, does playing in the major suit game gain or lose compared to 3NT?

"Pancake" hands are the flattest possible — 4333 distributions. These are the hands where Stayman is *least* expected to gain, because:
- The 4–4 major suit fit offers little ruffing value when both hands are completely flat.
- Declarer in a major suit contract must lose a trump trick on any 4–2 break, whereas 3NT has no such vulnerability.

We quantify the effect using a double-dummy simulation.

## Setup

- **South** is a 1NT opener: balanced 15–17 HCP (`any 4333 + any 4432 + any 5332`).
- **North** is a game-forcing responder with no slam interest: 4333 or 3433 shape, 9–14 HCP.
  - `4333`: North has exactly 4 spades (will bid Stayman looking for a spade fit).
  - `3433`: North has exactly 4 hearts (will bid Stayman looking for a heart fit).

### Auction outcomes

We restrict to deals where Stayman *always* finds a fit, so every deal is a direct comparison between 3NT and the major suit game.

| Strategy | Contract |
|---|---|
| **Raise** | 3NT–S always |
| **Stayman** | 4♥–S if both hold 4+ hearts; 4♠–S if both hold 4+ spades |

In [1]:
import bridgepandas as bp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

Unable to revert mtime: /Library/Fonts
Matplotlib is building the font cache; this may take a moment.


## 1. Generate deals

We use accept/reject sampling with scalar `Hand` properties for speed.
Generation takes roughly 30 seconds for 1000 deals.

In [2]:
_SOUTH_SHAPES   = {(5, 3, 3, 2), (4, 4, 3, 2), (4, 3, 3, 3)}  # any 5332 / 4432 / 4333
_NORTH_PATTERNS = {(4, 3, 3, 3), (3, 4, 3, 3)}                 # 4 spades or 4 hearts

def accept(deal):
    s, n = deal.south, deal.north
    return (
        s.shape in _SOUTH_SHAPES and 15 <= s.hcp <= 17
        and n.pattern in _NORTH_PATTERNS and 9 <= n.hcp <= 14
        and ((n.spades == 4 and s.spades >= 4) or (n.hearts == 4 and s.hearts >= 4))
    )

N = 1000
df = bp.random_deals(N, accept=accept, seed=42)
print(f"{len(df)} deals generated")

1000 deals generated


## 2. Verify constraints

In [3]:
pd.DataFrame({
    'south_hcp':    df['south'].hcp,
    'south_spades': df['south'].spades,
    'south_hearts': df['south'].hearts,
    'north_hcp':    df['north'].hcp,
    'north_spades': df['north'].spades,
    'north_hearts': df['north'].hearts,
}).describe().round(2)

,south_hcp,south_spades,south_hearts,north_hcp,north_spades,north_hearts
count,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
mean,15.74,3.25,3.17,10.85,3.52,3.48
std,0.77,0.84,0.86,1.53,0.5,0.5
min,15.0,2.0,2.0,9.0,3.0,3.0
25%,15.0,3.0,3.0,10.0,3.0,3.0
50%,16.0,3.0,3.0,11.0,4.0,3.0
75%,16.0,4.0,4.0,12.0,4.0,4.0
max,17.0,5.0,5.0,14.0,4.0,4.0


In [4]:
south_shapes = pd.Series([df.iloc[i].south.shape for i in range(N)])
print("South shapes:")
print(south_shapes.value_counts().to_string())
print()
north_has_4s = df['north'].spades == 4
print(f"North 4333 (4 spades): {north_has_4s.sum()}  ({north_has_4s.mean():.1%})")
print(f"North 3433 (4 hearts): {(~north_has_4s).sum()}  ({(~north_has_4s).mean():.1%})")

South shapes:
(4, 4, 3, 2)    433
(5, 3, 3, 2)    321
(4, 3, 3, 3)    246

North 4333 (4 spades): 522  (52.2%)
North 3433 (4 hearts): 478  (47.8%)


## 3. Determine Stayman contracts

North always has exactly one 4-card major. Stayman finds a fit if and only if South also holds 4+ cards in that suit.

In [5]:
heart_fit = (df['north'].hearts >= 4) & (df['south'].hearts >= 4)
spade_fit = (df['north'].spades >= 4) & (df['south'].spades >= 4)

stayman_contracts = pd.Series(
    np.where(heart_fit, '4H-S', '4S-S')
)

print("Stayman auction outcomes:")
print(f"  4\u2665\u2013S (heart fit): {heart_fit.sum():4d}  ({heart_fit.mean():.1%})")
print(f"  4\u2660\u2013S (spade fit): {spade_fit.sum():4d}  ({spade_fit.mean():.1%})")

Stayman auction outcomes:
  4♥–S (heart fit):  134  (13.4%)
  4♠–S (spade fit):  166  (16.6%)
  3NT–S (no fit):    700  (70.0%)


## 4. Double-dummy scoring

We use neither-vulnerable throughout. The `_dds` cache means the 3NT trick
count is computed once and reused when Stayman also lands in 3NT.

In [ ]:
VULN = '-'  # neither vulnerable

bp.add_dds(df, '3N-S',           'score_3nt',     VULN)
bp.add_dds(df, stayman_contracts, 'score_stayman', VULN)

df[['score_3nt', 'score_stayman']].head(10)

## 5. IMP differential

`imp_diff > 0` means 3NT scored better on that board; `imp_diff < 0` means Stayman scored better.

In [ ]:
df['imp_diff'] = (df['score_3nt'] - df['score_stayman']).map(bp.scorediff_imps)
df['outcome']  = np.where(heart_fit, '4H fit', '4S fit')

df[['score_3nt', 'score_stayman', 'imp_diff', 'outcome']].head(10)

## 6. Statistical analysis

In [ ]:
mean_diff       = df['imp_diff'].mean()
std_diff        = df['imp_diff'].std()
se              = std_diff / np.sqrt(N)
t_stat, p_value = stats.ttest_1samp(df['imp_diff'], 0)
ci_lo, ci_hi    = mean_diff - 1.96 * se, mean_diff + 1.96 * se

print("IMP differential  (positive = 3NT better, negative = Stayman better)")
print(f"  n         = {N}")
print(f"  mean      = {mean_diff:+.3f} IMPs")
print(f"  std dev   = {std_diff:.3f}")
print(f"  std error = {se:.3f}")
print(f"  95% CI    = ({ci_lo:+.3f}, {ci_hi:+.3f})")
print(f"  t         = {t_stat:.3f}")
print(f"  p-value   = {p_value:.4f}  (two-sided, H\u2080: mean = 0)")
print()
if p_value < 0.05:
    winner = '3NT' if mean_diff > 0 else 'Stayman'
    print(f"\u2192 {winner} is significantly better (p < 0.05).")
else:
    print(f"\u2192 No significant difference detected (p = {p_value:.3f}).")

## 7. Breakdown by auction outcome

In [ ]:
breakdown = (
    df.groupby('outcome')['imp_diff']
    .agg(count='count', mean='mean', std='std')
    .assign(se=lambda d: d['std'] / np.sqrt(d['count']))
)
breakdown[['count', 'mean', 'se']].round(3)

## 8. Distribution of IMP differentials

In [ ]:
df['imp_diff'] = (df['score_3nt'] - df['score_stayman']).map(bp.scorediff_imps)
df['outcome']  = np.where(heart_fit, '4H fit', '4S fit')

df[['score_3nt', 'score_stayman', 'imp_diff', 'outcome']].head(10)